In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Thiết lập đường dẫn & thư mục output

- `DATA_DIR`: nơi chứa 14 file CSV đầu vào.
- `OUTPUT_DIR`: nơi lưu 4 dataset đã làm sạch/merge.

In [3]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

In [4]:
CWD = Path(os.getcwd()).resolve()
REPO_ROOT = CWD if (CWD / "data").exists() else CWD.parent
DATA_DIR = REPO_ROOT / "data" / "datathon-2026-round-1"
OUTPUT_DIR = REPO_ROOT / "data" / "data_clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [5]:
colab_root = Path("/content/drive/MyDrive/Datathon_VinTelligence")
if colab_root.exists():
    REPO_ROOT = colab_root
    DATA_DIR = REPO_ROOT / "data" / "datathon-2026-round-1"
    OUTPUT_DIR = REPO_ROOT / "data" / "data_clean"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [16]:

if 'google.colab' in sys.modules:
    REPO_ROOT = Path("/content/drive/MyDrive/Datathon_VinTelligence")
else:
    CWD = Path(os.getcwd()).resolve()
    REPO_ROOT = CWD if (CWD / "data").exists() else CWD.parent
DATA_DIR = REPO_ROOT / "data" / "datathon-2026-round-1"
OUTPUT_DIR = REPO_ROOT / "data" / "data_clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [15]:
Files = {
    "sales": "sales.csv",
    "orders": "orders.csv",
    "order_items": "order_items.csv",
    "products": "products.csv",
    "promotions": "promotions.csv",
    "customers": "customers.csv",
    "reviews": "reviews.csv",
    "geography": "geography.csv",
    "shipments": "shipments.csv",
    "returns": "returns.csv",
    "inventory": "inventory.csv",
    "web_traffic": "web_traffic.csv",
    "payments": "payments.csv",
    "sample_submission": "sample_submission.csv",
}

missing = [f for f in Files.values() if not (DATA_DIR / f).exists()]
print("Missing files:", missing)
dfs = {}

for k, fname in Files.items():
    fpath = DATA_DIR / fname
    dfs[k] = pd.read_csv(fpath, low_memory=False)
    print(f"Loaded {k:<16} shape={dfs[k].shape} path={fpath}")
print("\nKeys loaded:", sorted(dfs.keys()))

Missing files: []
Loaded sales            shape=(3833, 3) path=/content/drive/MyDrive/Datathon_VinTelligence/data/datathon-2026-round-1/sales.csv
Loaded orders           shape=(646945, 8) path=/content/drive/MyDrive/Datathon_VinTelligence/data/datathon-2026-round-1/orders.csv
Loaded order_items      shape=(714669, 7) path=/content/drive/MyDrive/Datathon_VinTelligence/data/datathon-2026-round-1/order_items.csv
Loaded products         shape=(2412, 8) path=/content/drive/MyDrive/Datathon_VinTelligence/data/datathon-2026-round-1/products.csv
Loaded promotions       shape=(50, 10) path=/content/drive/MyDrive/Datathon_VinTelligence/data/datathon-2026-round-1/promotions.csv
Loaded customers        shape=(121930, 7) path=/content/drive/MyDrive/Datathon_VinTelligence/data/datathon-2026-round-1/customers.csv
Loaded reviews          shape=(113551, 7) path=/content/drive/MyDrive/Datathon_VinTelligence/data/datathon-2026-round-1/reviews.csv
Loaded geography        shape=(39948, 4) path=/content/dri

## Hàm cleaning dùng chung (tái sử dụng)

Khối hàm này gom các quy tắc bắt buộc:

- **Datetime standardizing**: mọi cột chứa `date` → datetime; dòng lỗi ngày tháng sẽ bị drop (có log).
- **Missing values**: `discount_amount=0`, `gender/age_group="Unknown"`.
- **Integrity**: `quantity>0`, `cogs<price`.
- **Duplicates**: drop trùng (ưu tiên theo khóa ở một số bảng).


In [8]:
def standardize_datetime_cols(df, hint = ""):
    df = df.copy()
    date_cols = [c for c in df.columns if "date" in c.lower()]
    if not date_cols:
        return df
    for c in date_cols:
        before_na = df[c].isna().sum()
        df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
        after_na = df[c].isna().sum()
        if after_na > before_na:
            print(f"[datetime] {hint} col={c} new_invalid_dates={(after_na - before_na):,}")
    mask_valid = df[date_cols].notna().all(axis=1)
    dropped = (~mask_valid).sum()
    if dropped:
        print(f"[datetime] {hint} dropping rows with invalid date(s): {dropped:,}")
        df = df.loc[mask_valid].copy()

    return df
def fill_missing_smart(df, hint = "") :
    df = df.copy()
    if "discount_amount" in df.columns:
        df["discount_amount"] = df["discount_amount"].fillna(0)

    for c in ["gender", "age_group"]:
        if c in df.columns:
            df[c] = df[c].fillna("Unknown")
    return df

def drop_duplicates_report(df, subset=None, hint="") :
    before = len(df)
    df2 = df.drop_duplicates(subset=subset)
    after = len(df2)
    return df2
def enforce_basic_integrity(df, subset=None, hint=""):
    df = df.copy()
    if "quantity" in df.columns:
        before = len(df)
        df = df.loc[df["quantity"].fillna(0) > 0].copy()
        if len(df) != before:
            print(f"[integrity] {hint} quantity>0 dropped={(before - len(df)):,}")
    if "cogs" in df.columns and "price" in df.columns:
        before = len(df)
        df = df.loc[df["cogs"].notna() & df["price"].notna() & (df["cogs"] < df["price"])].copy()
        if len(df) != before:
            print(f"[integrity] {hint} cogs<price dropped={(before - len(df)):,}")
    return df

## Cleaning phase

- Chuẩn hóa tất cả cột chứa `'date'` về datetime và loại các dòng lỗi ngày tháng.
- Missing values:
  - `discount_amount` → 0
  - `gender`, `age_group` → `'Unknown'`
- Integrity:
  - `quantity > 0`
  - `cogs < price`
- Duplicates: loại bản ghi trùng.


In [9]:
clean = {}
for name, df in dfs.items():
    print("\n" + "=" * 90)
    print(f"Table: {name} | original rows={len(df):,} cols={df.shape[1]}")
    df2 = standardize_datetime_cols(df, hint=name)
    df2 = fill_missing_smart(df2, hint=name)
    df2 = enforce_basic_integrity(df2, hint=name)
    subset = None
    if name == "orders" and "order_id" in df2.columns:
        subset = ["order_id"]
    if name == "customers" and "customer_id" in df2.columns:
        subset = ["customer_id"]
    if name == "products" and "product_id" in df2.columns:
        subset = ["product_id"]
    if name == "shipments" and "order_id" in df2.columns:
        subset = ["order_id"]
    df2 = drop_duplicates_report(df2, subset=subset, hint=name)
    clean[name] = df2


Table: sales | original rows=3,833 cols=3
Table: sales | cleaned rows=3,833 cols=3

Table: orders | original rows=646,945 cols=8


/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)


Table: orders | cleaned rows=646,945 cols=8

Table: order_items | original rows=714,669 cols=7
Table: order_items | cleaned rows=714,669 cols=7

Table: products | original rows=2,412 cols=8
Table: products | cleaned rows=2,412 cols=8

Table: promotions | original rows=50 cols=10
Table: promotions | cleaned rows=50 cols=10

Table: customers | original rows=121,930 cols=7
Table: customers | cleaned rows=121,930 cols=7

Table: reviews | original rows=113,551 cols=7
Table: reviews | cleaned rows=113,551 cols=7

Table: geography | original rows=39,948 cols=4
Table: geography | cleaned rows=39,948 cols=4

Table: shipments | original rows=566,067 cols=4


/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-dateti

Table: shipments | cleaned rows=566,067 cols=4

Table: returns | original rows=39,939 cols=7
Table: returns | cleaned rows=39,939 cols=7

Table: inventory | original rows=60,247 cols=17
Table: inventory | cleaned rows=60,247 cols=17

Table: web_traffic | original rows=3,652 cols=7
Table: web_traffic | cleaned rows=3,652 cols=7

Table: payments | original rows=646,945 cols=4


/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)


Table: payments | cleaned rows=646,945 cols=4

Table: sample_submission | original rows=548 cols=3
Table: sample_submission | cleaned rows=548 cols=3

Cleaning done. Tables: ['customers', 'geography', 'inventory', 'order_items', 'orders', 'payments', 'products', 'promotions', 'returns', 'reviews', 'sales', 'sample_submission', 'shipments', 'web_traffic']


/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)


## Cleaning phase (tạo dict `clean`)

Bước này chạy cleaning cho từng bảng:
- In ra `original rows/cols`.
- Chuẩn hóa datetime + xử lý missing + integrity.
- Drop duplicates và log số lượng record bị loại.

Kết quả là dict `clean` dùng xuyên suốt 4 bước build dataset.

In [10]:
sales = clean["sales"].copy()
orders = clean["orders"].copy()
items = clean["order_items"].copy()
products = clean["products"].copy()
promos = clean["promotions"].copy()

if "Date" in sales.columns:
    sales = sales.rename(columns={"Date": "date"})

sales = standardize_datetime_cols(sales, hint="sales")
orders = standardize_datetime_cols(orders, hint="orders")
promos = standardize_datetime_cols(promos, hint="promotions")
order_cols = ["order_id", "order_date", "order_status"]
for c in ["device_type", "order_source"]:
    if c in orders.columns:
        order_cols.append(c)
trx = items.merge(orders[order_cols], on="order_id", how="left")
trx = trx.merge(products[["product_id", "category", "segment", "price", "cogs"]], on="product_id", how="left")
if "promo_id" in trx.columns:
    trx = trx.merge(
        promos[["promo_id", "promo_type", "discount_value", "promo_channel", "stackable_flag"]],
        on="promo_id",
        how="left",
        suffixes=("", "_p1"),
    )
if "promo_id_2" in trx.columns:
    trx = trx.merge(
        promos[["promo_id", "promo_type", "discount_value", "promo_channel", "stackable_flag"]].rename(
            columns={
                "promo_id": "promo_id_2",
                "promo_type": "promo_type_2",
                "discount_value": "discount_value_2",
                "promo_channel": "promo_channel_2",
                "stackable_flag": "stackable_flag_2",
            }
        ),
        on="promo_id_2",
        how="left",
    )
trx["order_date"] = pd.to_datetime(trx["order_date"], errors="coerce")
trx = trx.loc[trx["order_date"].notna()].copy()
trx["date"] = trx["order_date"].dt.normalize()
if "device_type" in trx.columns:
    trx["is_mobile_order"] = trx["device_type"].astype(str).str.lower().str.contains("mobile", na=False).astype(int)
else:
    trx["is_mobile_order"] = 0

if "order_status" in trx.columns:
    trx["is_delivered"] = trx["order_status"].astype(str).str.lower().eq("delivered").astype(int)
else:
    trx["is_delivered"] = 0

if "segment" in trx.columns:
    trx["is_premium"] = trx["segment"].astype(str).str.lower().eq("premium").astype(int)
else:
    trx["is_premium"] = 0

if "order_source" in trx.columns:
    trx["order_source"] = trx["order_source"].fillna("unknown")
else:
    trx["order_source"] = "unknown"

promo_cols = [c for c in ["promo_id", "promo_id_2"] if c in trx.columns]
if promo_cols:
    trx["promo_used_flag"] = trx[promo_cols].notna().any(axis=1).astype(int)
else:
    trx["promo_used_flag"] = 0

stackable_1 = (trx.get("stackable_flag", 0) == 1) & trx.get("promo_id", pd.Series(False, index=trx.index)).notna()
stackable_2 = (trx.get("stackable_flag_2", 0) == 1) & trx.get("promo_id_2", pd.Series(False, index=trx.index)).notna()
trx["stackable_used_flag"] = (stackable_1 | stackable_2).astype(int)

order_level = trx.drop_duplicates("order_id")[
    ["order_id", "date", "is_mobile_order", "is_delivered", "order_source"]
].copy()
orders_daily = (
    order_level.groupby("date")
    .agg(
        orders_n=("order_id", "nunique"),
        mobile_orders=("is_mobile_order", "sum"),
        order_success_rate=("is_delivered", "mean"),
    )
    .reset_index()
)
lines_daily = (
    trx.groupby("date")
    .agg(
        items_n=("product_id", "count"),
        qty_sold=("quantity", "sum"),
        premium_qty=("quantity", lambda s: (s * trx.loc[s.index, "is_premium"]).sum()),
        total_discount_amount=("discount_amount", "sum"),
        total_lines=("order_id", "size"),
    )
    .reset_index()
)

lines_daily["premium_ratio"] = (
    lines_daily["premium_qty"] / lines_daily["qty_sold"].replace(0, np.nan)
).fillna(0)

daily = orders_daily.merge(lines_daily, on="date", how="left")
d1 = sales.merge(daily, on="date", how="left")

if "Revenue" in d1.columns and "COGS" in d1.columns:
    d1["gross_profit"] = d1["Revenue"] - d1["COGS"]
if "total_discount_amount" in d1.columns and "Revenue" in d1.columns:
    d1["discount_rate"] = (
        d1["total_discount_amount"].fillna(0) / d1["Revenue"].replace(0, np.nan)
    ).fillna(0)
if "Revenue" in d1.columns and "qty_sold" in d1.columns:
    d1["avg_item_price"] = (d1["Revenue"] / d1["qty_sold"].replace(0, np.nan)).fillna(0)
if "date" in d1.columns:
    d1["day_of_week"] = d1["date"].dt.day_name()
    d1["is_weekend"] = (d1["date"].dt.weekday >= 5).astype(int)
    d1["month"] = d1["date"].dt.month

for c in [
    "orders_n",
    "items_n",
    "qty_sold",
    "premium_qty",
    "total_discount_amount",
    "total_lines",
    "mobile_orders",
    "order_success_rate",
    "premium_ratio",
]:
    if c in d1.columns:
        d1[c] = d1[c].fillna(0)

out1 = OUTPUT_DIR / "Dataset_1_Revenue_Core.csv"
d1.to_csv(out1, index=False)
print("[D1] saved:", out1, "shape=", d1.shape)

d1.head()


/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-dateti

[D1] after + promotions(promo_id): (714669, 19)
[D1] after + promotions(promo_id_2): (714669, 23)
[D1] sales base rows: 3833
[D1] after merge sales x daily_trx rows: 3833
[D1] saved: /content/drive/MyDrive/Datathon_VinTelligence/data/data_clean/Dataset_1_Revenue_Core.csv shape= (3833, 18)


,date,Revenue,COGS,orders_n,mobile_orders,order_success_rate,items_n,qty_sold,premium_qty,total_discount_amount,total_lines,premium_ratio,gross_profit,discount_rate,avg_item_price,day_of_week,is_weekend,month
0,2012-07-04,5123547.94,3982991.19,162,69,0.827160,174,777,9,0.0,174,0.011583,1140556.75,0.0,6594.012793,Wednesday,0,7
1,2012-07-05,2751773.45,2150580.23,97,38,0.835052,103,428,13,0.0,103,0.030374,601193.22,0.0,6429.377220,Thursday,0,7
2,2012-07-06,3054029.42,2517632.84,93,42,0.741935,99,441,0,0.0,99,0.000000,536396.58,0.0,6925.236780,Friday,0,7
3,2012-07-07,2667930.94,2108246.62,73,34,0.753425,75,364,0,0.0,75,0.000000,559684.32,0.0,7329.480604,Saturday,1,7
4,2012-07-08,2360851.90,1808622.79,88,45,0.761364,94,394,0,0.0,94,0.000000,552229.11,0.0,5992.009898,Sunday,1,7


In [11]:
customers = clean["customers"].copy()
orders = clean["orders"].copy()
items = clean["order_items"].copy()
reviews = clean["reviews"].copy()
geog = clean["geography"].copy()

customers = standardize_datetime_cols(customers, hint="customers")
orders = standardize_datetime_cols(orders, hint="orders")
reviews = standardize_datetime_cols(reviews, hint="reviews")

order_amount = items.copy()
order_amount["line_net"] = order_amount["quantity"] * order_amount["unit_price"] - order_amount["discount_amount"].fillna(0)
order_amount = order_amount.groupby("order_id", as_index=False).agg(order_net_amount=("line_net", "sum"))
orders_enriched = orders.merge(order_amount, on="order_id", how="left")

cust_rfm = (
    orders_enriched.groupby("customer_id", as_index=False)
    .agg(
        Monetary=("order_net_amount", "sum"),
        Frequency=("order_id", "nunique"),
        Last_Order_Date=("order_date", "max"),
    )
)

cust_rating = reviews.groupby("customer_id", as_index=False).agg(Avg_Rating=("rating", "mean"), Reviews_N=("review_id", "nunique"))
d2 = customers.merge(geog, on="zip", how="left")
d2 = d2.merge(cust_rfm, on="customer_id", how="left")
d2 = d2.merge(cust_rating, on="customer_id", how="left")
print("[D2] after + rating rows:", len(d2))

for c in ["Monetary", "Frequency", "Avg_Rating", "Reviews_N"]:
    if c in d2.columns:
        d2[c] = d2[c].fillna(0)
out2 = OUTPUT_DIR / "Dataset_2_Customer_Insights.csv"
d2.to_csv(out2, index=False)
d2.head()

/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-dateti

[D2] order_amount rows: 646945
[D2] after orders x order_amount: (646945, 9)
[D2] customers base rows: 121930
[D2] after customers x geography rows: 121930
[D2] after + rfm rows: 121930
[D2] after + rating rows: 121930
[D2] saved: /content/drive/MyDrive/Datathon_VinTelligence/data/data_clean/Dataset_2_Customer_Insights.csv shape= (121930, 15)


,customer_id,zip,city_x,signup_date,gender,age_group,acquisition_channel,city_y,region,district,Monetary,Frequency,Last_Order_Date,Avg_Rating,Reviews_N
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,Hai Phong,East,District #13,142803.47,6.0,2021-04-24,0.0,0.0
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign,Hai Phong,East,District #13,204693.89,4.0,2022-07-06,2.0,2.0
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search,Hai Phong,East,District #13,52093.47,3.0,2013-07-29,5.0,1.0
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral,Hai Phong,East,District #13,10939.06,1.0,2020-06-28,0.0,0.0
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search,Hai Phong,East,District #13,64179.86,5.0,2019-03-27,5.0,1.0


In [12]:
ship = clean["shipments"].copy()
ret = clean["returns"].copy()
inv = clean["inventory"].copy()
orders = clean["orders"].copy()
items = clean["order_items"].copy()

ship = standardize_datetime_cols(ship, hint="shipments")
ret = standardize_datetime_cols(ret, hint="returns")
inv = standardize_datetime_cols(inv, hint="inventory")
orders = standardize_datetime_cols(orders, hint="orders")


ship["delivery_time_days"] = (ship["delivery_date"] - ship["ship_date"]).dt.days


ret_flag = ret.groupby("order_id", as_index=False).agg(return_flag=("return_id", "nunique"))
ret_flag["return_flag"] = (ret_flag["return_flag"] > 0).astype(int)


inv["snapshot_date"] = pd.to_datetime(inv["snapshot_date"], errors="coerce")
inv = inv.loc[inv["snapshot_date"].notna()].copy()
inv["inv_year"] = inv["snapshot_date"].dt.year
inv["inv_month"] = inv["snapshot_date"].dt.month

inv_monthly = (
    inv.groupby(["inv_year", "inv_month", "product_id"], as_index=False)
    .agg(
        stock_on_hand_avg=("stock_on_hand", "mean"),
        stockout_days_max=("stockout_days", "max"),
        fill_rate_avg=("fill_rate", "mean"),
        stockout_flag_max=("stockout_flag", "max")
    )
)


ship_items = ship.merge(orders[["order_id", "order_date"]], on="order_id", how="left")
ship_items = ship_items.merge(items[["order_id", "product_id", "quantity"]], on="order_id", how="left")
ship_items["ship_year"] = ship_items["ship_date"].dt.year
ship_items["ship_month"] = ship_items["ship_date"].dt.month


d3_temp = ship_items.merge(ret_flag, on="order_id", how="left")
d3_temp["return_flag"] = d3_temp["return_flag"].fillna(0).astype(int)
d3_temp = d3_temp.merge(
    inv_monthly,
    left_on=["ship_year", "ship_month", "product_id"],
    right_on=["inv_year", "inv_month", "product_id"],
    how="left",
)


d3 = d3_temp.groupby(
    ["order_id", "ship_date", "delivery_date", "shipping_fee", "delivery_time_days", "return_flag"],
    dropna=False,
    as_index=False
).agg(
    total_quantity=("quantity", "sum"),
    stock_on_hand_avg=("stock_on_hand_avg", "mean"),
    stockout_days_max=("stockout_days_max", "max"),
    fill_rate_avg=("fill_rate_avg", "mean"),
    stockout_flag_max=("stockout_flag_max", "max")
)
out3 = OUTPUT_DIR / "Dataset_3_Operations_Logistics.csv"
d3.to_csv(out3, index=False)
print("[D3] saved:", out3, "shape=", d3.shape)

d3.head()

/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-dateti

[D3] saved: /content/drive/MyDrive/Datathon_VinTelligence/data/data_clean/Dataset_3_Operations_Logistics.csv shape= (566067, 11)


,order_id,ship_date,delivery_date,shipping_fee,delivery_time_days,return_flag,total_quantity,stock_on_hand_avg,stockout_days_max,fill_rate_avg,stockout_flag_max
0,1,2012-07-07,2012-07-11,1.37,4,0,7,236.0,2.0,0.9333,1.0
1,2,2012-07-06,2012-07-10,2.60,4,1,7,236.0,2.0,0.9333,1.0
2,3,2012-07-04,2012-07-07,2.38,3,0,3,319.0,1.0,0.9667,1.0
3,4,2012-07-05,2012-07-11,2.49,6,0,5,327.0,0.0,1.0000,0.0
4,6,2012-07-09,2012-07-16,25.79,7,0,1,102.0,0.0,1.0000,0.0


In [13]:
traffic = clean["web_traffic"].copy()
orders = clean["orders"].copy()
items = clean["order_items"].copy()

traffic = standardize_datetime_cols(traffic, hint="web_traffic")
orders = standardize_datetime_cols(orders, hint="orders")
if "source" in traffic.columns and "traffic_source" not in traffic.columns:
    traffic = traffic.rename(columns={"source": "traffic_source"})
if "date" in traffic.columns:
    traffic["date"] = pd.to_datetime(traffic["date"], errors="coerce").dt.normalize()

order_amount = items.copy()
order_amount["line_net"] = (
    order_amount["quantity"] * order_amount["unit_price"]
    - order_amount["discount_amount"].fillna(0)
)
order_amount = (
    order_amount.groupby("order_id", as_index=False)
    .agg(order_net_amount=("line_net", "sum"))
)

orders2 = orders.merge(order_amount, on="order_id", how="left")
orders2["order_date"] = pd.to_datetime(orders2["order_date"], errors="coerce")
orders2 = orders2.loc[orders2["order_date"].notna()].copy()
orders2 = orders2.assign(
    date=orders2["order_date"].dt.normalize(),
    traffic_source=orders2["order_source"],
)

orders_daily_source = (
    orders2.groupby(["date", "traffic_source"], as_index=False)
    .agg(
        orders_n=("order_id", "nunique"),
        revenue_net=("order_net_amount", "sum"),
    )
)
d4 = traffic.merge(orders_daily_source, on=["date", "traffic_source"], how="left")
for c in ["orders_n", "revenue_net"]:
    d4[c] = d4[c].fillna(0)

d4["conversion_rate"] = (d4["orders_n"] / d4["sessions"].replace(0, np.nan)).fillna(0)
d4["AOV"] = (d4["revenue_net"] / d4["orders_n"].replace(0, np.nan)).fillna(0)
out4 = OUTPUT_DIR / "Dataset_4_Marketing_Traffic.csv"
d4.to_csv(out4, index=False)
print("[D4] saved:", out4, "shape=", d4.shape)

d4.head()

/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_2237/3346903400.py:8: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)


[D4] traffic base rows: 3652
[D4] after merge traffic x orders rows: 3652
[D4] saved: /content/drive/MyDrive/Datathon_VinTelligence/data/data_clean/Dataset_4_Marketing_Traffic.csv shape= (3652, 11)


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source,orders_n,revenue_net,conversion_rate,AOV
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search,80.0,1501578.07,0.008197,18769.725875
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search,18.0,766660.48,0.001721,42592.248889
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct,4.0,167151.91,0.000397,41787.977500
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct,5.0,130838.23,0.000501,26167.646000
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral,8.0,258038.00,0.000783,32254.750000
